# Receipt extraction with a fine-tuned small VLM: train + evaluate on a free T4

This notebook produces every GPU number in the README / reports:

1. zero-shot evaluation of `Qwen2.5-VL-3B-Instruct` on the CORD v2 **test** split
2. QLoRA fine-tuning on CORD v2 **train** (validation split for eval loss)
3. fine-tuned evaluation on the same test split, same prompt, same pipeline
4. robustness runs (rotation, blur, JPEG, small fonts, multi-page) and a resolution ablation
5. the API with the real model + the 5-parallel-request concurrency test
6. `reports/evaluation_report.md`

Runtime: *Runtime → Change runtime type → T4 GPU*. Total time is roughly 3–4 h; every step writes its results to disk, so a disconnect only costs the current step (training resumes from checkpoints with `--resume`).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

zsh:1: command not found: nvidia-smi


In [2]:
REPO_URL = "https://github.com/<your-user>/receipt-vlm-extraction.git"   # <- set
HF_ADAPTER_REPO = "<your-hf-user>/qwen25vl3b-cord-lora"                 # <- set
USE_DRIVE = True   # keep checkpoints/results on Google Drive across disconnects

import os
if not os.path.exists("receipt-vlm-extraction"):
    !git clone $REPO_URL receipt-vlm-extraction
%cd receipt-vlm-extraction
!pip install -q -r requirements-train.txt

zsh:1: no such file or directory: your-user
[Errno 2] No such file or directory: 'receipt-vlm-extraction'
/Users/sayantan/Projects/Document Retrieval App/notebooks


/Users/sayantan/Projects/Document Retrieval App/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:393: UserWarning: using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-train.txt'


In [3]:
OUT = "outputs/qwen25vl3b-cord-lora"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/receipt-vlm/qwen25vl3b-cord-lora"
    os.makedirs("/content/drive/MyDrive/receipt-vlm/results", exist_ok=True)
    !rm -rf results && ln -s /content/drive/MyDrive/receipt-vlm/results results
ADAPTER = f"{OUT}/adapter"
print(OUT)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Hugging Face token with write access (only needed to push the adapter)
from huggingface_hub import notebook_login
notebook_login()

## 0. Grounding audit (CPU only, no model)

In [4]:
!python -m scripts.grounding_audit --split test --threshold 85 --out results/grounding_audit.json | tail -5
!python -m scripts.grounding_audit --split test --threshold 75 --out results/grounding_audit_t75.json | tail -5
!python -m scripts.grounding_audit --split test --threshold 95 --out results/grounding_audit_t95.json | tail -5
# gold labels through the full pipeline: validates metrics + measures the recall cost of grounding
!python -m scripts.evaluate --oracle --name oracle_upper_bound | tail -5


/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.grounding_audit' (ModuleNotFoundError: No module named 'scripts')
/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.grounding_audit' (ModuleNotFoundError: No module named 'scripts')
/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.grounding_audit' (ModuleNotFoundError: No module named 'scripts')
/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.evaluate' (ModuleNotFoundError: No module named 'scripts')


## 1. Zero-shot baseline

In [5]:
!python -m scripts.evaluate --name zeroshot

/Users/sayantan/Projects/Document Retrieval App/.venv/bin/python: Error while finding module specification for 'scripts.evaluate' (ModuleNotFoundError: No module named 'scripts')


## 2. QLoRA fine-tuning
2 epochs over 800 receipts at batch 1 × grad-accum 8 = 200 optimizer steps. Re-run the same cell with `--resume` after a disconnect.

In [ ]:
!python -m training.train_lora --output-dir "$OUT" --epochs 2 --max-visual-tokens 1024 --push-to-hub "$HF_ADAPTER_REPO"

## 3. Fine-tuned evaluation (same split, prompt and pipeline as zero-shot)

In [ ]:
!python -m scripts.evaluate --name finetuned --adapter "$ADAPTER"

## 4. Failure modes and resolution trade-off

In [ ]:
for p in ["rotate5", "rotate90", "blur", "jpeg", "downscale", "multipage"]:
    !python -m scripts.evaluate --name finetuned_{p} --adapter "$ADAPTER" --perturb {p} | tail -3

In [ ]:
for vt in [256, 512, 1024, 1536]:
    !python -m scripts.evaluate --name finetuned_vt{vt} --adapter "$ADAPTER" --max-visual-tokens {vt} | tail -3

## 5. API with the real model + concurrency test
Starts uvicorn in the background with the fine-tuned adapter and fires 5 simultaneous requests.

In [ ]:
!python -m scripts.make_sample_pdfs --n 3 --out samples
import subprocess, time, requests
env = dict(os.environ, ADAPTER=ADAPTER, MAX_QUEUE="3", QUEUE_TIMEOUT_S="60", JOB_TIMEOUT_S="90")
server = subprocess.Popen(["uvicorn", "docextract.service.app:app", "--port", "8000", "--workers", "1"],
                          env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if requests.get("http://localhost:8000/readyz", timeout=2).ok: break
    except Exception: pass
    time.sleep(2)
print(requests.get("http://localhost:8000/readyz").json())

In [ ]:
!curl -s -F file=@samples/cord_test_000.pdf localhost:8000/extract | python -m json.tool | head -60

In [ ]:
!python -m scripts.concurrency_test --url http://localhost:8000 --pdf samples/cord_test_000.pdf --n 5 --save reports/concurrency_output.txt

In [ ]:
# bad-input behaviour against the real server
!head -c 100 /dev/urandom > bad.pdf && curl -s -F file=@bad.pdf localhost:8000/extract; echo
!printf '%%PDF-1.7 broken' > corrupt.pdf && curl -s -F file=@corrupt.pdf localhost:8000/extract; echo
!python -c "import fitz; d=fitz.open(); d.new_page(); d.save('locked.pdf', encryption=fitz.PDF_ENCRYPT_AES_256, user_pw='x', owner_pw='y')" && curl -s -F file=@locked.pdf localhost:8000/extract; echo
server.terminate()

## 6. Report

In [ ]:
!python -m scripts.make_report --results results --train-metrics "$OUT/train_metrics.json"
!cp "$OUT/train_metrics.json" results/ 2>/dev/null; zip -qr artifacts.zip results reports
from google.colab import files; files.download("artifacts.zip")